In [1]:
# Ensure jacopy is importable when this notebook is opened
# directly (not via pytest). Walks up from the notebook's
# directory to the repo root and prepends it to sys.path if
# jacopy isn't already installed into this kernel.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

# 02, The Jacobi identity

Companion notebook to [02_jacobi_identity.md](02_jacobi_identity.md). Closing the Jacobi identity for a Lie bracket in a single `prove_jacobi` call.

## The Lie bracket and three vector fields

`jacopy.brackets.lie.lie` is the module-level singleton for the standard manifold Lie bracket. The three vector fields are declared with the `VectorFields` helper, each gets `Graded(degree=0)` attached, which is what the Jacobi expansion consults for sign rules.

In [2]:
from jacopy import VectorFields
from jacopy.brackets.lie import lie
from jacopy.core.registry import PropertyRegistry

reg = PropertyRegistry()
X, Y, Z = VectorFields("X Y Z", registry=reg)

## `prove_jacobi`, does the obstruction vanish?

The graded Jacobi identity:

$$[X,[Y,Z]] + (-1)^{|X||Y|+|X||Z|}[Y,[Z,X]] + (-1)^{|Y||Z|+|X||Z|}[Z,[X,Y]] = 0$$

On the Lie bracket every degree is zero, so every sign is `+1`. `prove_jacobi(lie, X, Y, Z, registry=reg)` returns a `ProofChain` that simplifies the obstruction down to zero.

In [3]:
from jacopy.proof import prove_jacobi

chain = prove_jacobi(lie, X, Y, Z, registry=reg)
print('chain length:', len(chain))
print('final:', chain.steps[-1].after)

chain length: 2
final: 0


## Step-by-step rendering

The `display` layer renders `ProofChain`s as ASCII or LaTeX. Inside Jupyter, `display_chain` produces a LaTeX `align*` block that MathJax renders inline.

In [4]:
from jacopy.display import chain_to_ascii

print(chain_to_ascii(chain))

[bracket-expand] [X, [Y, Z]_{[·,·]}]_{[·,·]} + [Y, [Z, X]_{[·,·]}]_{[·,·]} + [Z, [X, Y]_{[·,·]}]_{[·,·]} -> (X * (Y * Z - Z * Y) - (Y * Z - Z * Y) * X) + (Y * (Z * X - X * Z) - (Z * X - X * Z) * Y) + (Z * (X * Y - Y * X) - (X * Y - Y * X) * Z)  -- expand bracket nodes by definition
[simplify] ((X * (Y * Z - Z * Y) - (Y * Z - Z * Y) * X) + (Y * (Z * X - X * Z) - (Z * X - X * Z) * Y) + (Z * (X * Y - Y * X) - (X * Y - Y * X) * Z)) - 0 -> 0  -- canonical-form pipeline


In [5]:
# Jupyter auto-renders the LaTeX (align* blocks):
from jacopy.display import display_chain
display_chain(chain)

{\allowdisplaybreaks\scriptsize
\begin{gather*}
\left[X,\, \left[Y,\, Z\right]_{[\cdot,\cdot]}\right]_{[\cdot,\cdot]} + \left[Y,\, \left[Z,\, X\right]_{[\cdot,\cdot]}\right]_{[\cdot,\cdot]} + \left[Z,\, \left[X,\, Y\right]_{[\cdot,\cdot]}\right]_{[\cdot,\cdot]} \to \left(X \, \left(Y \, Z - Z \, Y\right) - \left(Y \, Z - Z \, Y\right) \, X\right) + \left(Y \, \left(Z \, X - X \, Z\right) - \left(Z \, X - X \, Z\right) \, Y\right) + \left(Z \, \left(X \, Y - Y \, X\right) - \left(X \, Y - Y \, X\right) \, Z\right) \quad \text{[bracket-expand]}\;\text{--- expand bracket nodes by definition} \\
\left(\left(X \, \left(Y \, Z - Z \, Y\right) - \left(Y \, Z - Z \, Y\right) \, X\right) + \left(Y \, \left(Z \, X - X \, Z\right) - \left(Z \, X - X \, Z\right) \, Y\right) + \left(Z \, \left(X \, Y - Y \, X\right) - \left(X \, Y - Y \, X\right) \, Z\right)\right) - 0 \to 0 \quad \text{[simplify]}\;\text{--- canonical-form pipeline}
\end{gather*}
}

## Why does this work?

The Lie bracket's class definition sets `is_graded_antisymmetric=True` and `satisfies_graded_jacobi=True`. `prove_jacobi` first builds the obstruction, then simplifies it through the expansion engine's Jacobi recogniser, collapsing the residue to zero. For brackets with conditional Jacobi (such as `CourantBracket`) the same function does **not** drive the obstruction to zero; instead the returned chain records the condition (e.g. `dH = 0`) explicitly.

## Next step

Onwards to Poisson geometry: [03_poisson_geometry.md](03_poisson_geometry.md).